# Prefetch Assets

Interactive notebook for downloading **all** project datasets, RAG knowledge corpora,
and models. Assets are saved locally so you can inspect them before committing to DVC.

## Overview

| # | Section | Purpose | Source |
|---|---------|---------|--------|
| 1 | Training — Summarization | LoRA fine-tuning | `ccdv/arxiv-summarization` (HF) |
| 2 | Training — Code Generation | LoRA fine-tuning | `nvidia/OpenCodeInstruct` (HF) |
| 3 | Eval — Code | HumanEval benchmark | `openai/openai_humaneval` (HF) |
| 4 | Eval — QA | Natural Questions | `google-research-datasets/natural_questions` (HF) |
| 5 | Eval — QA | HotpotQA | `hotpot_qa` (HF) |
| 6 | Eval — Retrieval | MS MARCO | `ms_marco` (HF) |
| 7 | Eval — Retrieval | BEIR (subset) | `BeIR/*` (HF) |
| 8 | RAG — Chat | ArXiv papers | arXiv API |
| 9 | RAG — Code | PyTorch docs | Web scraping |
| 10 | Models | Base LLMs | HuggingFace Hub |

## How to use

1. Set `PROJECT_ROOT` to the root of your local clone.
2. Run the **Setup** cell.
3. Execute only the sections you need — each section is independent.
4. Inspect downloaded assets, then `dvc add` / `dvc push`.

> **Note:** Vector index building is done separately via
> `experiments/scripts/rag_data/build_vector_index.py` after the raw data is collected here.

---
## 0. Setup

In [1]:
from __future__ import annotations

import json
import time
import xml.etree.ElementTree as ET
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.parse import urlencode, urljoin
from urllib.request import Request, urlopen

from datasets import DatasetDict, load_dataset, load_from_disk
from huggingface_hub import snapshot_download

In [ ]:
import os

os.getcwd()

In [ ]:
# ---- Set the project root path for your machine ----
PROJECT_ROOT = Path("/home/jovyan")
# PROJECT_ROOT = Path("/home/anton-m/Git/agent-042")

ASSETS_DIR = PROJECT_ROOT / "assets"

assert PROJECT_ROOT.exists(), f"PROJECT_ROOT does not exist: {PROJECT_ROOT}"
print(f"Project root : {PROJECT_ROOT}")
print(f"Assets dir   : {ASSETS_DIR}")

---
## 0.1 Helper functions

In [ ]:
def save_hf_dataset(
    dataset_name: str,
    target_dir: Path,
    *,
    dataset_config: str | None = None,
    train_split: str | None = None,
    val_split: str | None = None,
) -> Path:
    """Download a HuggingFace dataset and save it to disk as Arrow files.

    Args:
        dataset_name: HuggingFace dataset identifier.
        target_dir: Local directory to save the dataset.
        dataset_config: Dataset configuration / subset name.
        train_split: Train split descriptor (supports slicing, e.g. 'train[:1%]').
        val_split: Validation split descriptor. Set to None to skip.

    Returns:
        Path to the saved dataset directory.
    """
    target_dir.mkdir(parents=True, exist_ok=True)
    print(
        f"Downloading {dataset_name}" + (f" [{dataset_config}]" if dataset_config else "") + " ..."
    )

    splits = {}
    if train_split is not None:
        splits["train"] = load_dataset(dataset_name, dataset_config, split=train_split)
    if val_split is not None:
        splits["validation"] = load_dataset(dataset_name, dataset_config, split=val_split)

    ds = DatasetDict(splits)
    print(f"Saving to disk: {target_dir}")
    ds.save_to_disk(str(target_dir))
    return target_dir


def inspect_hf_dataset(target_dir: Path) -> DatasetDict:
    """Load a previously saved HF dataset from disk and print basic stats."""
    ds = load_from_disk(str(target_dir))
    print(ds)
    for split_name, split_ds in ds.items():
        print(f"  {split_name}: {len(split_ds)} rows, columns={split_ds.column_names}")
    return ds

---
# Part I — Training Datasets

---

## 1. Summarization — `ccdv/arxiv-summarization`

LoRA fine-tuning for the summarization task.
Maps `article → abstract`. Using a **1 % subset** by default to keep download fast.

In [ ]:
ARXIV_SUMM_DIR = ASSETS_DIR / "datasets" / "arxiv-summarization"

save_hf_dataset(
    "ccdv/arxiv-summarization",
    ARXIV_SUMM_DIR,
    dataset_config="document",
    train_split="train",
    val_split="validation",
)
print(f"\n✅ arxiv-summarization saved to: {ARXIV_SUMM_DIR}")

In [ ]:
# Inspect
ds = inspect_hf_dataset(ARXIV_SUMM_DIR)
ds["train"][0]

## 2. Code Generation — `nvidia/OpenCodeInstruct`

LoRA fine-tuning for code generation.
Filtered to **Python** examples. Using a small subset by default.

In [ ]:
CODE_INSTRUCT_DIR = ASSETS_DIR / "datasets" / "open-code-instruct"

save_hf_dataset(
    "nvidia/OpenCodeInstruct",
    CODE_INSTRUCT_DIR,
    train_split="train",
    val_split=None,  # single-split dataset
)
print(f"\n✅ OpenCodeInstruct saved to: {CODE_INSTRUCT_DIR}")

In [ ]:
# Inspect
ds = inspect_hf_dataset(CODE_INSTRUCT_DIR)
ds["train"][0]

---
# Part II — Evaluation Datasets

---

## 3. Code Eval — `openai/openai_humaneval`

Measures executable rate and test-pass rate of generated code.

In [ ]:
HUMANEVAL_DIR = ASSETS_DIR / "datasets" / "humaneval"

save_hf_dataset(
    "openai/openai_humaneval",
    HUMANEVAL_DIR,
    train_split="test",  # HumanEval only has a "test" split
    val_split=None,
)
print(f"\n✅ HumanEval saved to: {HUMANEVAL_DIR}")

In [ ]:
# Inspect
ds = inspect_hf_dataset(HUMANEVAL_DIR)
ds["train"][0]

## 4. QA Eval — Natural Questions

Used for evaluating answer relevance and correctness (LLM-as-judge).
Downloading a **1 % subset** of the validation split.

In [ ]:
NQ_DIR = ASSETS_DIR / "datasets" / "natural-questions"

save_hf_dataset(
    "google-research-datasets/natural_questions",
    NQ_DIR,
    dataset_config="default",
    # train_split="train[:1%]",
    val_split="validation[:5%]",
)
print(f"\n✅ Natural Questions saved to: {NQ_DIR}")

In [ ]:
# Inspect
ds = inspect_hf_dataset(NQ_DIR)
ds["validation"][0]

## 5. QA Eval — HotpotQA

Multi-hop QA evaluation. Using the `distractor` setting and a small subset.

In [ ]:
HOTPOTQA_DIR = ASSETS_DIR / "datasets" / "hotpotqa"

save_hf_dataset(
    "hotpot_qa",
    HOTPOTQA_DIR,
    dataset_config="distractor",
    # train_split="train[:1%]",
    val_split="validation[:5%]",
)
print(f"\n✅ HotpotQA saved to: {HOTPOTQA_DIR}")

In [ ]:
# Inspect
ds = inspect_hf_dataset(HOTPOTQA_DIR)
ds["validation"][0]

## 6. Retrieval Eval — MS MARCO

Passage ranking benchmark. Used for Recall@k and nDCG@k evaluation.
Downloading a small validation subset.

In [ ]:
MSMARCO_DIR = ASSETS_DIR / "datasets" / "msmarco"

save_hf_dataset(
    "ms_marco",
    MSMARCO_DIR,
    dataset_config="v1.1",
    # train_split="train[:1%]",
    val_split="validation[:5%]",
)
print(f"\n✅ MS MARCO saved to: {MSMARCO_DIR}")

In [ ]:
# Inspect
ds = inspect_hf_dataset(MSMARCO_DIR)
ds["validation"][0]

## 7. Retrieval Eval — BEIR

BEIR is a heterogeneous benchmark for information retrieval.
We download individual sub-datasets relevant to the project (SciFact, NFCorpus).
Add more subsets as needed.

In [ ]:
BEIR_SUBSETS = ["scifact", "nfcorpus"]

for subset in BEIR_SUBSETS:
    subset_dir = ASSETS_DIR / "datasets" / f"beir-{subset}"
    subset_dir.mkdir(parents=True, exist_ok=True)
    print(f"Downloading BeIR/{subset} corpus ...")
    # BeIR repos use legacy dataset scripts unsupported in datasets>=3.0.
    # Load from the HF-maintained auto-converted parquet branch instead.
    ds = DatasetDict(
        {
            "train": load_dataset(
                "parquet",
                data_files=f"hf://datasets/BeIR/{subset}@refs/convert/parquet/corpus/corpus/0000.parquet",
                split="train",
            )
        }
    )
    ds.save_to_disk(str(subset_dir))
    print(f"✅ BEIR/{subset} saved to: {subset_dir}\n")

In [ ]:
# Inspect any BEIR subset
subset_dir = ASSETS_DIR / "datasets" / f"beir-{BEIR_SUBSETS[0]}"
ds = inspect_hf_dataset(subset_dir)
ds["train"][0]

### 7b. BEIR Queries + Qrels

Download the queries and relevance judgements (qrels) for each BEIR subset and
merge them with the existing corpus into a unified `DatasetDict`:

| split | content |
|---|---|
| `corpus` | all documents (`_id`, `title`, `text`) |
| `queries` | evaluation queries (`_id`, `text`) |
| `qrels` | relevance judgements (`query_id`, `corpus_id`, `score`) |

Qrels come from the companion `BeIR/<subset>-qrels` repo (test split).
Run this cell **after** cell 29 (corpus download).

In [ ]:
for subset in BEIR_SUBSETS:
    subset_dir = ASSETS_DIR / "datasets" / f"beir-{subset}"
    subset_dir.mkdir(parents=True, exist_ok=True)

    print(f"Downloading BeIR/{subset} queries ...")
    queries_ds = load_dataset(
        "parquet",
        data_files=f"hf://datasets/BeIR/{subset}@refs/convert/parquet/queries/queries/0000.parquet",
        split="train",
    )
    # Keep only the fields used by the eval runner
    queries_ds = queries_ds.select_columns(["_id", "text"])

    print(f"Downloading BeIR/{subset} qrels (test split) ...")
    qrels_ds = load_dataset(f"BeIR/{subset}-qrels", split="test")
    # Normalise column names to snake_case for consistency
    qrels_ds = qrels_ds.rename_columns({"query-id": "query_id", "corpus-id": "corpus_id"})

    # Load existing corpus split (already saved by cell 29)
    existing = DatasetDict.load_from_disk(str(subset_dir))
    corpus_ds = existing.get("train") or existing.get("corpus")

    merged = DatasetDict(
        {
            "corpus": corpus_ds,
            "queries": queries_ds,
            "qrels": qrels_ds,
        }
    )
    merged.save_to_disk(str(subset_dir))

    print(
        f"✅ BeIR/{subset} updated — "
        f"corpus: {len(corpus_ds)}, "
        f"queries: {len(queries_ds)}, "
        f"qrels: {len(qrels_ds)}\n"
    )

In [ ]:
# Verify the merged layout for the first subset
subset_dir = ASSETS_DIR / "datasets" / f"beir-{BEIR_SUBSETS[0]}"
ds = DatasetDict.load_from_disk(str(subset_dir))
print("Splits:", list(ds.keys()))
for split, split_ds in ds.items():
    print(f"  {split}: {split_ds.column_names} — {len(split_ds)} rows")
    print(f"    sample: {split_ds[0]}")

---
# Part III — RAG Knowledge Corpora

Raw data for the RAG vector store. After downloading here, build the vector
index with `experiments/scripts/rag_data/build_vector_index.py`.

---

## 8. Chat RAG — ArXiv Papers

Downloads recent ML/DL/AI papers (metadata + abstracts) from the arXiv search API.
The default query covers project-relevant areas: ML, AI, NLP, CV, IR, robotics,
neural nets, and statistical ML.

The helper first asks arXiv for the total number of matching papers, then downloads
only the requested slice with explicit pacing between requests.

For larger metadata harvests, prefer arXiv OAI-PMH instead of the search API.
For bulk PDFs or source files, arXiv provides requester-pays S3 buckets.

In [ ]:
# ---- ArXiv configuration ----
ARXIV_CATEGORIES = [
    "cs.LG",
    "stat.ML",
    "cs.AI",
    "cs.CL",
    "cs.CV",
    "cs.IR",
    "cs.NE",
    "cs.RO",
]
ARXIV_MAX_RESULTS = 2000  # Raise cautiously; large pulls are slow by design.
ARXIV_PAGE_SIZE = 100
ARXIV_DELAY_SECONDS = 4.0
ARXIV_MAX_RETRIES = 5
ARXIV_REQUEST_TIMEOUT = 60
ARXIV_OUTPUT_DIR = ASSETS_DIR / "rag_data" / "arxiv"

In [4]:
ARXIV_API_URL = "http://export.arxiv.org/api/query"
ARXIV_XML_NS = {
    "atom": "http://www.w3.org/2005/Atom",
    "opensearch": "http://a9.com/-/spec/opensearch/1.1/",
    "arxiv": "http://arxiv.org/schemas/atom",
}


def build_arxiv_query(categories: list[str]) -> str:
    """Build an arXiv category query that matches any of the requested subjects."""
    return "(" + " OR ".join(f"cat:{category}" for category in categories) + ")"


def _get_arxiv_setting(name: str, fallback: int | float) -> int | float:
    return globals().get(name, fallback)


def _parse_retry_after(retry_after: str | None) -> float | None:
    if retry_after is None:
        return None

    try:
        return max(float(retry_after), 0.0)
    except ValueError:
        return None


def _fetch_arxiv_feed(
    search_query: str,
    *,
    start: int,
    max_results: int,
    request_timeout: int,
    max_retries: int,
    base_delay_seconds: float,
) -> ET.Element:
    params = urlencode(
        {
            "search_query": search_query,
            "start": start,
            "max_results": max_results,
            "sortBy": "submittedDate",
            "sortOrder": "descending",
        }
    )
    request = Request(
        f"{ARXIV_API_URL}?{params}",
        headers={"User-Agent": "agent-042-prefetch-assets/1.0"},
    )

    backoff_seconds = max(base_delay_seconds, 1.0)
    for attempt in range(1, max_retries + 1):
        try:
            with urlopen(request, timeout=request_timeout) as response:
                return ET.fromstring(response.read())
        except HTTPError as exc:
            should_retry = exc.code in {429, 500, 502, 503, 504} and attempt < max_retries
            if not should_retry:
                raise

            sleep_seconds = _parse_retry_after(exc.headers.get("Retry-After")) or backoff_seconds
            print(
                f"  HTTP {exc.code} from arXiv; "
                f"sleeping {sleep_seconds:.1f}s before retry {attempt}/{max_retries}"
            )
            time.sleep(sleep_seconds)
            backoff_seconds *= 2
        except URLError as exc:
            if attempt == max_retries:
                raise

            print(
                f"  Network error from arXiv ({exc.reason}); "
                f"sleeping {backoff_seconds:.1f}s before retry {attempt}/{max_retries}"
            )
            time.sleep(backoff_seconds)
            backoff_seconds *= 2

    raise RuntimeError("arXiv request retry loop exhausted unexpectedly")


def get_arxiv_total_results(
    categories: list[str],
    *,
    request_timeout: int | None = None,
    max_retries: int | None = None,
    base_delay_seconds: float | None = None,
) -> int:
    """Return the total number of search results for the configured category query."""
    request_timeout = int(request_timeout or _get_arxiv_setting("ARXIV_REQUEST_TIMEOUT", 60))
    max_retries = int(max_retries or _get_arxiv_setting("ARXIV_MAX_RETRIES", 5))
    base_delay_seconds = float(base_delay_seconds or _get_arxiv_setting("ARXIV_DELAY_SECONDS", 4.0))

    query = build_arxiv_query(categories)
    feed = _fetch_arxiv_feed(
        query,
        start=0,
        max_results=0,
        request_timeout=request_timeout,
        max_retries=max_retries,
        base_delay_seconds=base_delay_seconds,
    )
    total_results = feed.findtext("opensearch:totalResults", default="0", namespaces=ARXIV_XML_NS)
    return int(total_results)


def _normalize_arxiv_text(value: str | None) -> str:
    return " ".join((value or "").split())


def _parse_arxiv_entry(entry: ET.Element) -> dict:
    entry_id = entry.findtext("atom:id", default="", namespaces=ARXIV_XML_NS)
    primary_category = entry.find("arxiv:primary_category", ARXIV_XML_NS)

    pdf_url = None
    for link in entry.findall("atom:link", ARXIV_XML_NS):
        if link.attrib.get("title") == "pdf":
            pdf_url = link.attrib.get("href")
            break

    return {
        "arxiv_id": entry_id.rsplit("/", 1)[-1],
        "title": _normalize_arxiv_text(
            entry.findtext("atom:title", default="", namespaces=ARXIV_XML_NS)
        ),
        "authors": [
            _normalize_arxiv_text(author.findtext("atom:name", default="", namespaces=ARXIV_XML_NS))
            for author in entry.findall("atom:author", ARXIV_XML_NS)
        ],
        "abstract": _normalize_arxiv_text(
            entry.findtext("atom:summary", default="", namespaces=ARXIV_XML_NS)
        ),
        "published": entry.findtext("atom:published", default="", namespaces=ARXIV_XML_NS),
        "updated": entry.findtext("atom:updated", default="", namespaces=ARXIV_XML_NS),
        "categories": [
            category.attrib["term"]
            for category in entry.findall("atom:category", ARXIV_XML_NS)
            if category.attrib.get("scheme") == ARXIV_XML_NS["arxiv"]
        ],
        "primary_category": primary_category.attrib.get("term")
        if primary_category is not None
        else None,
        "pdf_url": pdf_url,
    }


def download_arxiv_papers(
    categories: list[str],
    max_results: int,
    output_dir: Path,
    *,
    page_size: int | None = None,
    delay_seconds: float | None = None,
    max_retries: int | None = None,
    request_timeout: int | None = None,
) -> list[dict]:
    """Download paper metadata from arXiv with explicit paging and polite pacing.

    Args:
        categories: arXiv categories to include in the query.
        max_results: Maximum number of papers to fetch.
        output_dir: Directory to write arxiv_papers.json.
        page_size: Per-request page size sent to the search API.
        delay_seconds: Delay between successful page requests.
        max_retries: Maximum retry attempts for rate limits and transient failures.
        request_timeout: Per-request timeout in seconds.

    Returns:
        List of paper metadata dicts.
    """
    page_size = int(page_size or _get_arxiv_setting("ARXIV_PAGE_SIZE", 100))
    delay_seconds = float(delay_seconds or _get_arxiv_setting("ARXIV_DELAY_SECONDS", 4.0))
    max_retries = int(max_retries or _get_arxiv_setting("ARXIV_MAX_RETRIES", 5))
    request_timeout = int(request_timeout or _get_arxiv_setting("ARXIV_REQUEST_TIMEOUT", 60))

    output_dir.mkdir(parents=True, exist_ok=True)

    query = build_arxiv_query(categories)
    total_available = get_arxiv_total_results(
        categories,
        request_timeout=request_timeout,
        max_retries=max_retries,
        base_delay_seconds=delay_seconds,
    )
    target_results = min(max_results, total_available)

    print(f"Searching arXiv: {query}")
    print(f"Matched {total_available} papers across {len(categories)} categories.")
    print(
        f"Downloading {target_results} papers "
        f"with page_size={page_size} and delay={delay_seconds:.1f}s between requests."
    )

    papers: list[dict] = []
    for start in range(0, target_results, page_size):
        batch_size = min(page_size, target_results - start)
        feed = _fetch_arxiv_feed(
            query,
            start=start,
            max_results=batch_size,
            request_timeout=request_timeout,
            max_retries=max_retries,
            base_delay_seconds=delay_seconds,
        )
        entries = feed.findall("atom:entry", ARXIV_XML_NS)
        if not entries:
            print(f"  No entries returned at offset {start}; stopping early.")
            break

        papers.extend(_parse_arxiv_entry(entry) for entry in entries)
        print(f"  fetched {len(papers)}/{target_results} papers")

        if len(entries) < batch_size:
            print("  arXiv returned fewer records than requested; stopping early.")
            break

        if len(papers) < target_results:
            time.sleep(delay_seconds)

    output_file = output_dir / "arxiv_papers.json"
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(papers, f, indent=2, ensure_ascii=False)

    metadata_file = output_dir / "arxiv_fetch_metadata.json"
    with open(metadata_file, "w", encoding="utf-8") as f:
        json.dump(
            {
                "query": query,
                "categories": categories,
                "total_available": total_available,
                "requested_results": max_results,
                "downloaded_results": len(papers),
                "page_size": page_size,
                "delay_seconds": delay_seconds,
            },
            f,
            indent=2,
            ensure_ascii=False,
        )

    print(f"Downloaded {len(papers)} papers -> {output_file}")
    print(f"Saved fetch metadata -> {metadata_file}")
    return papers

In [ ]:
arxiv_papers = download_arxiv_papers(ARXIV_CATEGORIES, ARXIV_MAX_RESULTS, ARXIV_OUTPUT_DIR)
print(f"\n✅ ArXiv papers saved to: {ARXIV_OUTPUT_DIR}")

In [ ]:
# Inspect
print(f"Total papers: {len(arxiv_papers)}")
print(f"Example title: {arxiv_papers[0]['title']}")
print(f"Categories: {arxiv_papers[0]['categories']}")
print(f"Abstract (first 300 chars): {arxiv_papers[0]['abstract'][:300]}...")

## 9. Code RAG — PyTorch Documentation

Scrapes a curated set of PyTorch documentation pages (core API, tutorials).
Saved as JSON for later chunking and indexing.

In [ ]:
# ---- PyTorch docs configuration ----
PYTORCH_BASE_URL = "https://pytorch.org/docs/stable/"
PYTORCH_OUTPUT_DIR = ASSETS_DIR / "rag_data" / "pytorch_docs"

# Core API pages to scrape (extend as needed)
PYTORCH_PAGES = [
    "generated/torch.nn.Module.html",
    "generated/torch.Tensor.html",
    "generated/torch.nn.Linear.html",
    "generated/torch.nn.Conv2d.html",
    "generated/torch.nn.functional.relu.html",
    "generated/torch.optim.Adam.html",
    "generated/torch.optim.SGD.html",
    "generated/torch.nn.CrossEntropyLoss.html",
    "generated/torch.nn.MSELoss.html",
    "generated/torch.autograd.backward.html",
    "tensors.html",
    "autograd.html",
    "nn.html",
    "optim.html",
    "torch.html",
]

In [ ]:
import sys

src_path = PROJECT_ROOT / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from shared.pytorch_docs_scraper import scrape_pytorch_doc_page as _scrape_pytorch_doc_page


def collect_pytorch_docs(
    base_url: str,
    page_list: list[str],
    output_dir: Path,
) -> list[dict]:
    """Scrape a list of PyTorch doc pages and save to JSON."""
    output_dir.mkdir(parents=True, exist_ok=True)
    pages = []
    for i, page_path in enumerate(page_list, 1):
        url = urljoin(base_url, page_path)
        print(f"[{i}/{len(page_list)}] {url}")
        try:
            page, skip_reason = _scrape_pytorch_doc_page(url, max_code_examples=10)
            if page is None:
                print(f"  ⚠ Skipping page: {skip_reason}")
            else:
                pages.append(page)
            time.sleep(1)  # be nice to the server
        except Exception as e:
            print(f"  ⚠ Error: {e}")

    output_file = output_dir / "pytorch_docs.json"
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(pages, f, indent=2, ensure_ascii=False)

    print(f"\nScraped {len(pages)} pages → {output_file}")
    return pages

In [ ]:
pytorch_pages = collect_pytorch_docs(PYTORCH_BASE_URL, PYTORCH_PAGES, PYTORCH_OUTPUT_DIR)
print(f"\n✅ PyTorch docs saved to: {PYTORCH_OUTPUT_DIR}")

In [ ]:
# Inspect
print(f"Total pages: {len(pytorch_pages)}")
for p in pytorch_pages[:3]:
    print(f"  • {p['title']}  ({len(p['content'])} chars, {len(p['code_examples'])} code blocks)")

---
# Part IV — Models

---

## 10. Prefetch Model

Download a base LLM from HuggingFace Hub.

Available presets:
- `Qwen/Qwen3-0.6B` (default)
- `ministral/Ministral-3b-instruct`
- `mistralai/Mistral-7B-v0.1`

In [ ]:
# ---- Model configuration ----
MODEL_ID = "Qwen/Qwen3-0.6B"
MODEL_TARGET_DIR = ASSETS_DIR / "models" / MODEL_ID

In [ ]:
def download_model(model_id: str, target_dir: Path) -> Path:
    """Download a HuggingFace model repo snapshot to a local directory."""
    target_dir.mkdir(parents=True, exist_ok=True)
    print(f"Downloading model repo {model_id} to {target_dir} ...")
    local_repo = snapshot_download(
        repo_id=model_id,
        local_dir=str(target_dir),
        local_dir_use_symlinks=False,
    )
    return Path(local_repo)

In [ ]:
model_path = download_model(MODEL_ID, MODEL_TARGET_DIR)
print(f"\n✅ Model stored at: {model_path}")

In [ ]:
# Inspect downloaded model files
for f in sorted(MODEL_TARGET_DIR.rglob("*")):
    if f.is_file():
        size_mb = f.stat().st_size / (1024 * 1024)
        print(f"  {f.relative_to(MODEL_TARGET_DIR)}  ({size_mb:.1f} MB)")

---
# DVC — Commit Assets

After you are happy with the downloaded assets, track them with DVC:

```bash
# Training datasets
dvc add assets/datasets/arxiv-summarization-01
dvc add assets/datasets/open-code-instruct-01

# Evaluation datasets
dvc add assets/datasets/humaneval
dvc add assets/datasets/natural-questions-01
dvc add assets/datasets/hotpotqa-01
dvc add assets/datasets/msmarco-01
dvc add assets/datasets/beir-scifact
dvc add assets/datasets/beir-nfcorpus

# RAG corpora
dvc add assets/rag_data/arxiv
dvc add assets/rag_data/pytorch_docs

# Models
dvc add assets/models/Qwen/Qwen3-0.6B

dvc push
```

 > **Next step (RAG only):** Build vector indices by running
 > `python experiments/scripts/rag_data/build_vector_index.py`.